# Semana 2 - Limpeza de Dados

**Carga horaria:** 10h  
**Entregavel:** Dataset Iris limpo e validado

## Objetivos
- Gerar um dataset com problemas reais.
- Tratar valores ausentes, duplicatas e inconsistencias.
- Validar criterios de qualidade de dados.

In [ ]:
from pathlib import Path
import pandas as pd

from dirty_iris import criar_iris_sujo

In [ ]:
df_dirty = criar_iris_sujo(nivel='medio', random_state=42)
print('Shape sujo:', df_dirty.shape)
display(df_dirty.head())
display(df_dirty.isna().sum())
print('Duplicatas:', df_dirty.duplicated().sum())

## Regras de limpeza
1. Padronizar nomes de especies para: setosa, versicolor, virginica.
2. Remover duplicatas.
3. Tratar ausentes numericos com mediana.
4. Tratar ausentes em especie com moda.
5. Remover medidas negativas (fisicamente invalidas).

In [ ]:
def normalize_species(value):
    if pd.isna(value):
        return value
    txt = str(value).strip().lower()
    mapping = {
        'setosa': 'setosa', 'setoza': 'setosa',
        'versicolor': 'versicolor', 'versicolour': 'versicolor',
        'virginica': 'virginica', 'virginic': 'virginica', 'virginica': 'virginica',
        'virginica': 'virginica', 'virginica ': 'virginica'
    }
    txt = txt.replace('í', 'i')
    return mapping.get(txt, txt)

df = df_dirty.copy()
df['species_name'] = df['species_name'].apply(normalize_species)

numeric_cols = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove invalidos fisicos
for col in numeric_cols:
    df = df[df[col].isna() | (df[col] >= 0)]

# Remove duplicatas
df = df.drop_duplicates().reset_index(drop=True)

# Preenche ausentes
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

mode_species = df['species_name'].mode(dropna=True)[0]
df['species_name'] = df['species_name'].fillna(mode_species)

print('Shape limpo:', df.shape)
display(df.head())

In [ ]:
# Validacao do entregavel
expected_species = {'setosa', 'versicolor', 'virginica'}

assert df.isna().sum().sum() == 0, 'Ainda existem valores ausentes.'
assert df.duplicated().sum() == 0, 'Ainda existem duplicatas.'
assert set(df['species_name'].unique()).issubset(expected_species), 'Especies fora do dominio esperado.'

for col in numeric_cols:
    assert (df[col] >= 0).all(), f'Valores negativos encontrados em {col}'

print('Entregavel Semana 2 OK: dataset Iris limpo e validado.')

In [ ]:
output = Path('data/iris_limpo.csv')
output.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output, index=False)
print('Arquivo salvo em', output)